# 정수 계산 확인 실습

S3 다음에 진행하는 실습입니다. [A6 정수 눈금](https://github.com/tsi03048/resnet20-accel/blob/main/docs/fundamentals/A6.md)과 [D1 반올림 규칙](https://github.com/tsi03048/resnet20-accel/blob/main/docs/fundamentals/D1.md)을 읽고 돌아옵니다.


저장된 출력은 기본 경로를 점검한 실행 예시입니다. 본인 결과는 준비 셀부터 다시 실행해 확인합니다. 전체 데이터 평가와 실제 보드 실행은 이 예시에 포함하지 않았습니다.

## 이번에 비교할 것

같은 입력과 Weight·Bias를 쓰고, 같은 반올림 규칙을 적용해 미리 저장한 정답(Golden)과 비교합니다. Stem은 모델 맨 앞 Conv, Main은 계산하는 길, Skip은 원래 값을 보내는 길입니다.

이 실행은 소프트웨어의 중간 결과와 출력을 비교합니다. 회로 실행과 실제 보드 검사는 뒤의 별도 단계입니다.


## 실행 전 짧은 예측

Main code 8·Scale 1/8과 Skip code 4·Scale 1/4의 합을 출력 Scale 1/8로 표현하면 12일까요, 16일까요?

![정수와 눈금 맞추기](https://raw.githubusercontent.com/tsi03048/resnet20-accel/main/docs/figures/reading-v1/A6.png)

실제 Stem 첫 출력에서는 곱의 합 411에 Bias 11879를 더해 중간 합(acc) 12290을 얻습니다. 출력 눈금으로 바꾸는 곱셈값(multiplier)은 3519126, 2의 거듭제곱으로 나누는 자리 수(shift)는 31입니다. 따라서 12290×3519126÷(2^31)을 RNE로 반올림하면 20입니다. 여기서 2^31은 2를 31번 곱한 수입니다.

[실제 Stem 첫 출력 그림](https://github.com/tsi03048/resnet20-accel/blob/main/docs/learning/images/integer_units.png)


## 준비와 실행

준비 셀은 폴더와 NumPy를 확인합니다. 그다음 셀은 파일이 같은 묶음인지 검사하고 정수 계산을 다시 실행합니다. 학습이나 전체 사진 다운로드는 하지 않습니다. 완료될 때까지 기다린 뒤 결과를 읽습니다.

In [1]:
# 0. E1/E2-A 실습 폴더와 분리해 준비한다.
import os, sys, subprocess, json, importlib.util
from pathlib import Path
repo=Path('/content/resnet20-accel-e2b') if Path('/content').is_dir() else (Path.cwd() if (Path.cwd()/'quant/replay_e2b.py').exists() else Path.cwd()/'resnet20-accel-e2b')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/tsi03048/resnet20-accel.git',str(repo)],check=True)
os.chdir(repo)
if not Path('quant/replay_e2b.py').exists():
    raise RuntimeError('이 폴더는 이전 버전입니다. 작업을 저장하고 새 런타임/폴더에서 다시 시작하세요.')
sys.path.insert(0,str(repo))
if importlib.util.find_spec('numpy') is None:
    subprocess.run([sys.executable,'-m','pip','install','numpy'],check=True)
print('실행 폴더:', repo)
if (repo/'.git').exists():
    print('소스 버전:', subprocess.run(['git','rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip())
else:
    print('ZIP/local copy: 사용한 배포 ZIP 이름을 기록하세요.')


실행 폴더: [프로젝트 복사본: 로컬 경로 생략]
ZIP/local copy: 사용한 배포 ZIP 이름을 기록하세요.


In [2]:
# 1. 파일 검사를 통과한 상수와 입력으로 실제 정수 계산
from quant.replay_e2b import replay
from datetime import datetime, timezone
result=replay(backend='numpy')
result['executed_at_utc']=datetime.now(timezone.utc).isoformat()
Path('results/e2b_student_replay.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print(json.dumps(result,indent=2))


{
  "status": "PASS",
  "backend": "numpy",
  "tensor_count": 61,
  "compared_elements": 552084,
  "prediction": 3,
  "class_name": "cat",
  "package_semantic_sha256": "f375f8b33e87e750d8e7e6ef90a49368d03684f22a706525e0083d061890b833",
  "rtl_compared": false,
  "executed_at_utc": "2026-11-02T09:19:14.924471+00:00"
}


정상 결과: PASS(비교 통과) / 중간 출력 61곳에서 숫자 552,084개 일치 / 예측 후보 번호 3 / 고양이(cat).
이것은 미리 저장한 정답(Golden)과 소프트웨어 결과의 비교이며 RTL·보드 통과가 아니다.


### 결과 항목과 다를 때 확인할 곳

- tensor_count=61: 입력을 포함한 비교 지점 수입니다.
- compared_elements=552084: 비교한 숫자 총개수입니다.
- prediction=3, class_name=cat: 가장 큰 점수의 후보입니다.
- rtl_compared=false: 회로 실행은 포함하지 않았습니다.

hash·identity·shape 오류는 먼저 파일 묶음·크기를 확인합니다. first mismatch 뒤의 이름은 처음 달라진 비교 지점입니다. 기대값 파일을 고치지 말고 입력·계수·반올림 순서를 봅니다. [자세한 확인 순서](https://github.com/tsi03048/resnet20-accel/blob/main/docs/learning/S3.md)

다음 셀은 저장된 첫 중간 합에 후처리 규칙을 직접 적용합니다. Conv 전체를 이 셀에서 다시 계산하는 것은 아닙니다.

In [3]:
# 2. 실제 Stem 첫 원소를 따라가기
import numpy as np
from quant.integer_math import rne_div
with np.load('reference/e2b_v1/constants.npz',allow_pickle=False) as z:
    bias=int(z['stem.bias'][0]); n=int(z['stem.multiplier'][0]); shift=int(z['stem.shift'][0])
with np.load('reference/e2b_v1/sample0/tensors.npz',allow_pickle=False) as z:
    acc=int(z['stem.acc'][0,0,0,0]); expected=int(z['stem'][0,0,0,0])
code=min(127,max(0,rne_div(acc*n,1<<shift)))
print('곱의 합:',acc-bias,'Bias:',bias,'acc:',acc,'N:',n,'shift:',shift)
print('직접 계산:',code,'Golden:',expected)
if code!=expected: raise RuntimeError('첫 Stem code 불일치')


곱의 합: 411 Bias: 11879 acc: 12290 N: 3519126 shift: 31
직접 계산: 20 Golden: 20


### 첫 출력 결과 읽기

곱의 합 411 → Bias 11879를 더한 acc 12290 → N=3519126, shift=31 적용 → 직접 계산 20과 Golden 20 순서로 읽습니다. N은 출력 눈금으로 바꾸는 곱셈값입니다. acc는 같은데 결과가 다르면 RNE·shift·ReLU·포화 순서를 확인합니다.

## 관찰 직후 확인

1. 위 Stem 출력 20은 어떤 누산값과 계수에서 나왔나요?
2. 2의 거듭제곱으로 나누며 자릿수를 줄이는 Shift와, 중간값에서 짝수를 택하는 반올림 RNE는 왜 다를 수 있나요?
3. 한 장의 61개 비교 지점이 일치하면, 10,000장 정확도와 실제 회로 속도까지 확인했다고 할 수 있나요?

In [4]:
from quant.integer_math import rne_div
print('동점 반올림:', [rne_div(v,2) for v in (5,7,-5,-7)])
print('기대값: [2,4,-2,-4]')

동점 반올림: [2, 4, -2, -4]
기대값: [2,4,-2,-4]


### 반올림 확인

결과 [2,4,-2,-4]를 입력 2.5·3.5·-2.5·-3.5와 같은 순서로 읽습니다. 정확히 중간인 값에서 짝수를 택하는지 봅니다. 여기까지가 기본 한 장 실습입니다. 아래 전체 평가의 RUN_FULL은 처음에는 False로 둡니다.

## 선택: 상수부터 재생성하고 Test 10,000장 평가

아래 `RUN_FULL`을 True로 바꾸면 학습된 숫자를 담은 체크포인트 파일과 CIFAR 사진 자료를 받고 전체 평가한다.
배포한 E2-A 출처를 고정해 읽으며 새로 선택하지 않는다. CPU에서 시간이 걸린다.
빠른 16장 점검은 전체 정확도 결과로 제출하지 않는다.


In [5]:
RUN_FULL=False
if RUN_FULL:
    if any(importlib.util.find_spec(p) is None for p in ('torch','torchvision','datasets')):
        subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
    subprocess.run([sys.executable,'-m','quant.run_e2b','--limit','10000','--audit','8','--batch-size','32'],check=True)
    print(Path('results/e2b_integer.json').read_text(encoding='utf-8'))
else:
    print('한 장 재현만 실행했습니다. results/e2b_integer.json은 저장소에 배포된 기준 실행 기록입니다.')


한 장 재현만 실행했습니다. results/e2b_integer.json은 저장소에 배포된 기준 실행 기록입니다.


## 제출

실행자·날짜·사용한 파일 묶음의 식별값(hash)·실행한 범위·첫 Stem 계산·확인 질문 답안을 함께 제출한다.
ZIP에는 이번 한 장 재현 기록과 사용한 패키지 설명 파일을 넣습니다. 전체 평가를 직접 실행한 경우에만 그 실행 기록을 추가합니다.


In [6]:
# 3. 결과·상수·Golden 보관
from zipfile import ZipFile, ZIP_DEFLATED
with ZipFile('e2b_outputs.zip','w',ZIP_DEFLATED) as z:
    items = ['results/e2b_student_replay.json', 'reference/e2b_v1/manifest.json',
             'reference/e2b_v1/sample0/manifest.json']
    if RUN_FULL:
        items += ['results/e2b_run_manifest.json', 'results/e2b_integer.json', 'results/e2b_samples.csv']
    for item in items:
        path = Path(item)
        if path.is_file(): z.write(path)
try:
    from google.colab import files
    files.download('e2b_outputs.zip')
except ImportError:
    print(Path('e2b_outputs.zip').resolve())


[프로젝트 복사본: 로컬 경로 생략]\e2b_outputs.zip
